# 01. Collecte des données

Sentinelle Scolaire, défi 1 : cartographie et anticipation du risque de décrochage scolaire.

Cette étape rassemble, sans les modifier, toutes les données utiles au défi 1 parmi celles remises par les organisateurs :

- la base principale DGESS 2016-2025 (`DONNEES_STATISTQUES_ PRESCOLAIRE ET PRIMAIRE_2016_2025_DCB.xlsx`) ;
- le Data Pack DCB 2026 : annuaires et tableaux de bord DGESS, rapports ST-ESU, annuaires régionaux INSD, déplacés internes (CONASUR), conditions de vie, limites administratives COD-AB, écoles OpenStreetMap.

Principes :

- les fichiers des organisateurs (`DataPack_DCB_2026/`, base principale, document du challenge) sont en lecture seule ; le code refuse d'y écrire ;
- les lignes extraites sont copiées telles quelles, avec leur origine (fichier, page, ligne) ; la remise en forme relève de l'étape 02 ;
- chaque fichier source est identifié par son empreinte SHA-256 et son URL d'origine.

Sorties : `data/01_collecte/` (extraits, registre, inventaire, contrôle de couverture) et `depots/01-collecte/` (fichiers nommés selon la convention du challenge).

Le code est dans `src/sentinelle/collecte/` ; ce notebook l'exécute et en présente les résultats.

In [1]:
import sys
from pathlib import Path

RACINE = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(RACINE / "src"))

import pandas as pd

from sentinelle import config
from sentinelle.collecte import CATALOGUE, executer_collecte
from sentinelle.depots import exporter_collecte

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)

## 1. Jeux de données retenus

Le catalogue déclare, pour chaque jeu, les fichiers du pack concernés et la règle de sélection des pages :

- « tout » : le fichier entier ;
- « entete » : les pages dont l'en-tête (quatre premières lignes) porte l'intitulé de la section, ce qui reste stable d'une année à l'autre alors que les numéros de page changent ;
- « texte » : les pages dont une ligne porte l'intitulé, hors table des matières ; utilisé pour les annuaires régionaux INSD dont la numérotation des tableaux diffère d'une région à l'autre.

In [2]:
catalogue = pd.DataFrame(
    {
        "jeu": j.identifiant,
        "libelle": j.libelle,
        "niveau_geo": j.niveau_geo,
        "regle": j.regle.mode if j.regle.mode == "tout" else f"{j.regle.mode} : {j.regle.motif}",
        "usage": j.usage,
    }
    for j in CATALOGUE
)
catalogue

,jeu,libelle,niveau_geo,regle,usage
0,dgess_flux_province,"Taux de promotion, redoublement et abandon par province, classe et sexe",province,"entete : TAUX DE PROMOTION, REDOUBLEMENT",Variable cible : taux d'abandon par province
1,dgess_effectifs_redoublants_province,Historique des effectifs et des redoublants par niveau et par province,province,entete : HISTORIQUE DES EFFECTIFS ET DES REDOUBLANTS,Recalcul des taux de flux par la méthode de la cohorte reconstituée
2,dgess_effectifs_historique_province,Historique des effectifs par province,province,entete : HISTORIQUE DES EFFECTIFS PAR PROVINCE,"Évolution des effectifs, chocs d'effectifs"
3,dgess_principaux_chiffres_province,"Écoles, salles de classe, élèves, groupes pédagogiques et maîtres par province",province,"entete : NOMBRE D.ECOLES.{0,5}SALLES","Offre scolaire : élèves par classe, élèves par maître"
4,dgess_taux_acces_province,"Taux d'admission, de scolarisation et d'achèvement par province",province,entete : TAUX (?:BRUT|NET) D|TAUX D.ACH,"Accès et achèvement, analysés à part de l'abandon"
5,dgess_distance_province,Pourcentage des élèves selon la distance parcourue par province,province,entete : DISTANCE,Éloignement de l'école
6,dgess_ecoles_salles_province,Répartition des écoles et salles de classe publiques et privées par province,province,entete : REPARTITION DES ECOLES ET (?:DES )?SALLES,Offre scolaire
7,dgess_cep_province,Résultats au certificat d'études primaires par province,province,entete : CERTIFICAT,Résultats scolaires
8,dgess_communes_prioritaires,Principaux chiffres des 43 communes prioritaires (annuaires 2017-2018 à 2020...,commune,entete : COMMUNES PRIORITAIRES|^PRIORITAIRES,"Indicateurs communaux DGESS, dont flux, sur une partie du territoire"
9,dgess_tableaux_de_bord_primaire,Tableaux de bord de l'enseignement primaire 2015-2016 à 2023-2024,region,tout,"Séries régionales d'abandon, de redoublement et d'achèvement ; contrôle des ..."


## 2. Exécution de la collecte

In [3]:
resultats = executer_collecte()
registre = resultats["registre"]
couverture = resultats["couverture"]
inventaire = resultats["inventaire"]

print(f"{registre['jeu'].nunique()} jeux, {len(registre)} fichiers sources, "
      f"{registre['nb_lignes'].astype(int).sum():,} lignes extraites".replace(",", " "))

33 jeux  191 fichiers sources  312 727 lignes extraites


## 3. Inventaire des fichiers remis par les organisateurs

Chaque fichier du dossier des organisateurs est décrit : format, taille, empreinte, dimensions pour les CSV, producteur et URL d'après le manifeste du pack.

In [4]:
(inventaire
 .assign(bloc=inventaire["dossier"].str.replace("DataPack_DCB_2026/data/", "data/", regex=False).str.split("/").str[:2].str.join("/"))
 .groupby(["nature", "bloc"])
 .agg(fichiers=("fichier", "count"), taille_mo=("taille_ko", lambda s: round(s.sum() / 1024, 1)))
 .reset_index())

,nature,bloc,fichiers,taille_mo
0,extraction CSV du pack,data/00_Documentation_et_Referentiels,2,0.1
1,extraction CSV du pack,data/01_Education_MENAPLN_DGESS,89,55.2
2,extraction CSV du pack,data/02_Demographie_INSD,32,42.0
3,extraction CSV du pack,data/03_Conditions_de_Vie_et_Emploi,43,13.7
4,extraction CSV du pack,data/04_Cartographie_SIG,8,2.0
5,fichier source,.,2,0.5
6,fichier source,DataPack_DCB_2026,2,0.5
7,fichier source,DataPack_DCB_2026/00_Documentation_et_Referentiels,4,0.8
8,fichier source,DataPack_DCB_2026/01_Education_MENAPLN_DGESS,89,433.3
9,fichier source,DataPack_DCB_2026/02_Demographie_INSD,32,199.0


In [5]:
inventaire.groupby("format").agg(fichiers=("fichier", "count"), taille_mo=("taille_ko", lambda s: round(s.sum() / 1024, 1)))

,fichiers,taille_mo
format,,
csv,176,114.5
html,1,0.0
json,1,0.0
parquet,1,1.2
pdf,138,684.4
txt,1,0.0
xlsx,23,7.3
zip,6,60.8


### Bilan : sources retenues et écartées

Chaque fichier source remis par les organisateurs reçoit un statut. « Retenue » : au moins un jeu l'utilise. « Écartée » : le motif est indiqué. « À examiner » : aucune décision n'a encore été prise, ce qui ne doit pas arriver.

In [6]:
bilan = resultats["bilan"]
bilan["statut"].value_counts()

statut
retenue    118
écartée     55
Name: count, dtype: int64

In [7]:
(bilan[bilan["statut"] == "écartée"]
 .groupby("motif_exclusion")
 .agg(fichiers=("chemin", "count"))
 .sort_values("fichiers", ascending=False))

,fichiers
motif_exclusion,
Éducation non formelle : hors du champ du défi (primaire formel),22
"Géométries et métadonnées cartographiques : utilisées à l'étape de cartographie, pas de tableau à collecter",9
Préscolaire : hors du champ du défi (primaire),6
"Documentation du challenge et du Data Pack : lue, sert à la traçabilité",5
Résultats généraux du RGPH 2019 : chiffres repris plus finement par les autres publications retenues,4
Annuaires des ONG : interventions non reliées à la cible ; pistes de leviers pour plus tard,2
"Documentation du RGPH : définitions citées, aucune donnée à extraire",2
"Caractéristiques des ménages (RGPH 2019) : niveau national et régional, redondant avec l'EHCVM",1
"Projections RGPH (NADA) : même contenu que le volume 4 des projections, retenu",1


In [8]:
bilan[bilan["statut"] == "à examiner"]

,chemin,format,taille_ko,statut,jeux,motif_exclusion


## 4. Registre des sources

Une ligne par fichier source utilisé : jeu, période, pages retenues, nombre de lignes extraites, empreinte, producteur, URL d'origine et date de collecte.

In [9]:
(registre
 .assign(nb_lignes=registre["nb_lignes"].astype(int))
 .groupby(["jeu", "niveau_geo"])
 .agg(fichiers=("source_fichier", "count"),
      periodes=("periode", lambda s: ", ".join(sorted(set(p for p in s if p)))[:60]),
      lignes=("nb_lignes", "sum"))
 .reset_index())

,jeu,niveau_geo,fichiers,periodes,lignes
0,bm_unesco_series_nationales,pays,1,1970-2025,15290
1,codab_communes,commune,1,2025-08,352
2,codab_provinces,province,1,2025-08,48
3,codab_regions,region,1,2025-08,18
4,dgess_base_principale,region,1,2015-2025,1568
5,dgess_cep_province,province,8,"2017-2018, 2018-2019, 2019-2020, 2020-2021, 2021-2022, 2022-",667
6,dgess_communes_prioritaires,commune,4,"2017-2018, 2018-2019, 2019-2020, 2020-2021",2408
7,dgess_depliants_primaire,pays,7,"2017-2018, 2018-2019, 2019-2020, 2020-2021, 2021-2022, 2022-",645
8,dgess_distance_province,province,8,"2017-2018, 2018-2019, 2019-2020, 2020-2021, 2021-2022, 2022-",3515
9,dgess_ecoles_salles_province,province,8,"2017-2018, 2018-2019, 2019-2020, 2020-2021, 2021-2022, 2022-",2322


In [10]:
registre[["jeu", "source_fichier", "periode", "pages_retenues", "nb_lignes", "producteur", "url"]].head(12)

,jeu,source_fichier,periode,pages_retenues,nb_lignes,producteur,url
0,dgess_base_principale,DONNEES_STATISTQUES_ PRESCOLAIRE ET PRIMAIRE_2016_2025_DCB.xlsx,2015-2025,toutes,1568,DGESS / MENAPLN,
1,dgess_flux_province,CNS521_Annuaire_statistique_de_lenseignement_Primaire_2017_2018.pdf,2017-2018,43-47,182,CNS / DGESS,https://www.cns.bf/publications/telecharger/521/
2,dgess_flux_province,CNS522_Annuaire_statistique_de_lenseignement_Primaire_2018_2019.pdf,2018-2019,43-47,221,CNS / DGESS,https://www.cns.bf/publications/telecharger/522/
3,dgess_flux_province,CNS523_Annuaire_statistique_de_lenseignement_Primaire_2019_2020.pdf,2019-2020,40-44,221,CNS / DGESS,https://www.cns.bf/publications/telecharger/523/
4,dgess_flux_province,CNS524_Annuaire_statistique_de_lenseignement_Primaire_2020_2021.pdf,2020-2021,40-44,221,CNS / DGESS,https://www.cns.bf/publications/telecharger/524/
5,dgess_flux_province,CNS525_Annuaire_statistique_de_lenseignement_Primaire_2021_2022.pdf,2021-2022,27-31,221,CNS / DGESS,https://www.cns.bf/publications/telecharger/525/
6,dgess_flux_province,CNS526_Annuaire_statistique_de_lenseignement_Primaire_2022_2023.pdf,2022-2023,27-31,222,CNS / DGESS,https://www.cns.bf/publications/telecharger/526/
7,dgess_flux_province,CNS529_Annuaire_statistique_de_lenseignement_primaire_2023_2024.pdf,2023-2024,27-31,222,CNS / DGESS,https://www.cns.bf/publications/telecharger/529/
8,dgess_flux_province,CNS530_Annuaire_statistique_de_lenseignement_primaire_2024_2025.pdf,2024-2025,29-33,222,CNS / DGESS,https://www.cns.bf/publications/telecharger/530/
9,dgess_effectifs_redoublants_province,CNS521_Annuaire_statistique_de_lenseignement_Primaire_2017_2018.pdf,2017-2018,404-432,2536,CNS / DGESS,https://www.cns.bf/publications/telecharger/521/


## 5. Contrôle de couverture

Pour les jeux sélectifs, on vérifie que chaque fichier fournit des pages et que le volume extrait est comparable d'une année à l'autre. Un écart marqué signale une section absente, un intitulé différent ou une extraction incomplète, à vérifier sur le PDF.

In [11]:
annuaires = couverture[couverture["jeu"].str.startswith("dgess_")].copy()
annuaires["nb_lignes"] = annuaires["nb_lignes"].astype(int)
annuaires.pivot_table(index="jeu", columns="periode", values="nb_lignes", aggfunc="sum", fill_value=0)

periode,2017-2018,2018-2019,2019-2020,2020-2021,2021-2022,2022-2023,2023-2024,2024-2025
jeu,,,,,,,,
dgess_cep_province,82,83,83,84,84,83,85,83
dgess_communes_prioritaires,693,589,577,549,0,0,0,0
dgess_distance_province,440,440,440,440,440,435,440,440
dgess_ecoles_salles_province,293,293,293,293,293,283,287,287
dgess_effectifs_historique_province,164,169,169,169,169,161,161,161
dgess_effectifs_redoublants_province,2536,2544,2776,2772,2773,2536,2536,2540
dgess_flux_province,182,221,221,221,221,222,222,222
dgess_principaux_chiffres_province,234,234,234,234,234,231,234,234
dgess_taux_acces_province,341,342,342,343,343,343,343,429


In [12]:
ecarts = annuaires.assign(
    mediane=annuaires.groupby("jeu")["nb_lignes"].transform("median"),
)
ecarts["ecart_relatif"] = (ecarts["nb_lignes"] / ecarts["mediane"] - 1).round(2)
ecarts.loc[ecarts["ecart_relatif"].abs() > 0.15, ["jeu", "periode", "pages_retenues", "nb_lignes", "mediane", "ecart_relatif"]]

,jeu,periode,pages_retenues,nb_lignes,mediane,ecart_relatif
1,dgess_flux_province,2017-2018,43-47,182,221.0,-0.18
40,dgess_taux_acces_province,2024-2025,19-28,429,343.0,0.25
65,dgess_communes_prioritaires,2017-2018,12-27,693,583.0,0.19


In [13]:
couverture.loc[couverture["alerte"] != "", ["jeu", "source_fichier", "alerte"]]

,jeu,source_fichier,alerte


In [14]:
registre.loc[registre["periode"] == "", ["jeu", "source_fichier"]]

,jeu,source_fichier


## 6. Aperçu : taux de promotion, redoublement et abandon par province, 2024-2025

Extrait brut, tel qu'il figure dans l'annuaire : une ligne par indicateur, le nom de province sur la ligne du redoublement, les colonnes CP1 à CM2 par sexe (G, F, T).

In [15]:
flux = pd.read_csv(config.SORTIE_COLLECTE / "extraits" / "dgess_flux_province.csv",
                   sep=";", dtype=str, encoding="utf-8-sig", keep_default_na=False)
flux.loc[flux["source_periode"] == "2024-2025", ["page", "ligne"] + [f"c{i}" for i in range(1, 9)]].head(15)

,page,ligne,c1,c2,c3,c4,c5,c6,c7,c8
1510,29,1,BF/MEBAPLN/SG/DGESS,Année scolaire 2024-2025,,,,,,
1511,29,2,I. LESPRINCIPAUX CHIFFRES,,,,,,,
1512,29,3,"I. 8. TAUX DE PROMOTION, REDOUBLEMENT ET ABANDON",,,,,,,
1513,29,4,CP1,CP2,CE1,CE2,CM1,CM2,,
1514,29,5,G,F,T,G,F,T,G,F
1515,29,6,BOUCLE DU MOUHOUN,,,,,,,
1516,29,7,TX Prom,"99,4","99,4","99,4","97,0","97,2","97,1","98,2"
1517,29,8,BANWA,TX Red,"0,6","0,6","0,6","3,0","2,8","2,9"
1518,29,9,TX Aban,"0,0","0,0","0,0","0,0","0,0","0,0","0,0"
1519,29,10,TX Prom,"90,3","91,3","90,8","92,5","93,4","92,9","87,8"


## 7. Base principale DGESS : feuilles et tableaux

In [16]:
base = pd.read_csv(config.SORTIE_COLLECTE / "extraits" / "dgess_base_principale.csv",
                   sep=";", dtype=str, encoding="utf-8-sig", keep_default_na=False)
titres = base[base["col1"].str.match(r"(?i)^tableau")][["feuille", "ligne", "col1"]]
titres.rename(columns={"col1": "tableau"})

,feuille,ligne,tableau
0,PRESCOLAIRE,1,Tableau 1 : Évolution des effectifs du préscolaire de 2015/2016 à 2024/2025 ...
18,PRESCOLAIRE,19,Tableau 2: Évolution des effectifs du préscolaire de 2015/2016 à 2024/2025 p...
26,PRESCOLAIRE,27,Tableau 3: Évolution des effectifs du préscolaire de 2015/2016 à 2024/2025 p...
35,PRESCOLAIRE,36,Tableau 4: Évolution des effectifs du préscolaire de 2015/2016 à 2024/2025 p...
53,PRESCOLAIRE,54,Tableau 5: Évolution du TBS au préscolaire de 2015/2016 à 2024/2025 par sexe...
72,PRESCOLAIRE,73,Tableau 6: Évolution de l’IPS du TBS au préscolaire de 2015/2016 à 2024/2025...
90,PRESCOLAIRE,91,Tableau 7 : Évolution du nombre de structures éducatives du préscolaire de 2...
147,PRESCOLAIRE,148,Tableau 8 : Évolution du nombre de salles de classes du préscolaire de 2015/...
235,Feuil1,1,Tableau 1 : Évolution des effectifs du primaire de 2015/2016 à 2024/2025 par...
302,Feuil2,1,Tableau 2: Évolution des effectifs du primaire de 2015/2016 à 2024/2025 par ...


## 8. Dépôts au format du challenge

Les sorties sont copiées dans `depots/01-collecte/` sous la forme `EQUIPE_01-collecte_D1_engineer_description_vN.csv`, avec un LISEZ-MOI. Le code équipe se règle par la variable d'environnement `SENTINELLE_EQUIPE`.

In [17]:
depot = exporter_collecte(version=1)
print("Code équipe :", config.EQUIPE)
depot

Code équipe : EXX


,fichier_depose,role,fichier_produit,contenu
0,EXX_01-collecte_D1_engineer_registre-sources_v1.csv,engineer,data/01_collecte/registre_sources.csv,"Registre des sources : fichier d'origine, pages retenues, empreinte SHA-256,..."
1,EXX_01-collecte_D1_engineer_bilan-sources-retenues-ecartees_v1.csv,engineer,data/01_collecte/bilan_sources.csv,Statut de chaque fichier remis par les organisateurs : retenu (avec les jeux...
2,EXX_01-collecte_D1_engineer_inventaire-fichiers-organisateurs_v1.csv,engineer,data/01_collecte/inventaire_fichiers.csv,"Inventaire de tous les fichiers remis par les organisateurs (format, taille,..."
3,EXX_01-collecte_D1_engineer_controle-couverture-extractions_v1.csv,engineer,data/01_collecte/controle_couverture.csv,"Pages retenues par fichier pour les jeux sélectifs, avec alerte si aucune pa..."
4,EXX_01-collecte_D1_engineer_series-nationales-banque-mondiale-1970-2025_v1.csv,engineer,data/01_collecte/extraits/bm_unesco_series_nationales.csv,"Indicateurs éducation Banque mondiale et UNESCO, séries nationales"
5,EXX_01-collecte_D1_engineer_referentiel-communes-pcodes-13-17-regions_v1.csv,engineer,data/01_collecte/extraits/codab_communes.csv,"Limites administratives COD-AB : communes, avec anciens et nouveaux P-codes"
6,EXX_01-collecte_D1_engineer_referentiel-provinces-pcodes_v1.csv,engineer,data/01_collecte/extraits/codab_provinces.csv,Limites administratives COD-AB : provinces
7,EXX_01-collecte_D1_engineer_referentiel-regions-pcodes_v1.csv,engineer,data/01_collecte/extraits/codab_regions.csv,Limites administratives COD-AB : régions
8,EXX_01-collecte_D1_engineer_base-principale-dgess-region-2016-2025_v1.csv,engineer,data/01_collecte/extraits/dgess_base_principale.csv,Données statistiques du préscolaire et du primaire 2016-2025 (base principale)
9,EXX_01-collecte_D1_engineer_resultats-cep-province-2017-2025_v1.csv,engineer,data/01_collecte/extraits/dgess_cep_province.csv,Résultats au certificat d'études primaires par province


## 9. Limites et suite

- Les extraits restent au format des PDF : en-têtes sur plusieurs lignes, noms de région en titre de bloc, lignes de total, virgule décimale. Leur mise en forme en tables longues est l'objet de l'étape 02.
- Les rapports ST-ESU scannés ont été lus par reconnaissance de caractères (colonne `methode` = « OCR - a verifier ») : leurs chiffres doivent être contrôlés sur les PDF.
- Les taux d'abandon publiés valent souvent 0,0 : l'abandon étant calculé par différence d'effectifs, les valeurs négatives semblent ramenées à zéro. L'historique des effectifs et des redoublants par province permettra de les recalculer.
- Les annuaires régionaux INSD manquent pour le Centre-Est, le Centre-Nord et l'Est.
- Aucune donnée n'est disponible au niveau de l'école ou de l'élève.